# בניית אפליקציות ליצירת תמונות (OpenAI)

יש ל-LLM יותר מיצירת טקסט. ניתן גם ליצור תמונות מתיאורי טקסט. תמונות כמדיום שימושיות בתחומי מדטק, אדריכלות, תיירות, פיתוח משחקים, שיווק ועוד. בשיעור זה אנו עובדים עם מודלי **GPT Image** העכשוויים בפלטפורמת OpenAI.

## יעדי למידה

בסוף שיעור זה תוכל ל:

- להסביר מהי יצירת תמונות ואיפה היא שימושית.
- להבין את משפחת המודלים `gpt-image` וכיצד היא שונה מהמודלים הישנים של DALL·E.
- לבנות אפליקציית יצירת תמונות ולערוך תמונות.

## מהי יצירת תמונות?

מודלי יצירת תמונות יוצרים תמונות מהנחית טקסט. מודלים מודרניים כמו `gpt-image` לומדים את הקשר בין טקסט לתמונות במהלך האימון, ואז בהדרגה הופכים רעש אקראי לתמונה שמתאימה להנחייה שלך.

שתי משפחות מודלים מוכרות ליצירת תמונות הן:

- **`gpt-image` (OpenAI)** — הדור הנוכחי המשמש בשיעור זה. הוא תומך ביצירת תמונה מטקסט ועריכת תמונות (inpainting עם מסכה).
- **Midjourney** — מודל צד שלישי פופולרי עם שירות משלו וזרימת עבודה מבוססת Discord.

> מודלי התמונות הישנים של OpenAI — **DALL·E 2** ו-**DALL·E 3** — הם מיושנים. DALL·E 3 כבר אינו זמין לפריסות חדשות, ותכונות כמו `create_variation` היו קיימות רק ב-DALL·E 2. השתמש במודלי `gpt-image` לאפליקציות חדשות.

> **חשוב:** מודלי `gpt-image` מחזירים את התמונה שנוצרה כ-**base64** (`b64_json`), לא כממשק URL. הקוד שלך מפענח את מחרוזת ה-base64 לבייטים ושומר אותה — אין כתובת URL לתמונה להורדה.


## בניית אפליקציית יצירת תמונות ראשונה שלך

אז מה דרוש כדי לבנות אפליקציית יצירת תמונות? אתה צריך את הספריות הבאות:

- **python-dotenv**, מומלץ מאוד להשתמש בספריה זו כדי לשמור את הסודות שלך בקובץ *.env* נפרד מהקוד.
- **openai**, ספריה זו היא מה שתשתמש בו כדי לתקשר עם ממשק ה-API של OpenAI.
- **pillow**, לעבודה עם תמונות בפייתון.


1. צור קובץ *.env* עם התוכן הבא:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. אספו את הספריות שלמעלה בקובץ בשם *requirements.txt* כך:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. לאחר מכן, צרו סביבה וירטואלית והתקינו את הספריות:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> עבור Windows, השתמש בפקודות הבאות כדי ליצור ולהפעיל את סביבת העבודה הווירטואלית שלך:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. הוסף את הקוד הבא בקובץ שנקרא *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # ייבא dotenv
    dotenv.load_dotenv()

    # צור אובייקט OpenAI (קורא את OPENAI_API_KEY מתוך קובץ ה-.env שלך)
    client = openai.OpenAI()


    try:
        # צור תמונה באמצעות ממשק API ליצירת תמונות
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # הזן את טקסט ההנחיה שלך כאן
            size='1024x1024',
            n=1
        )
        # הגדר את התיקייה לשמירת התמונה
        image_dir = os.path.join(os.curdir, 'images')

        # אם התיקייה לא קיימת, צור אותה
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # אתחל את הנתיב של התמונה (שים לב שסוג הקובץ צריך להיות png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # דגמי gpt-image מחזירים את התמונה כ-base64 (b64_json), לא כ-URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # הצג את התמונה בצופה התמונות המוגדר כברירת מחדל
        image = Image.open(image_path)
        image.show()

    # לטפל בשגיאות יוצאות דופן
    except openai.BadRequestError as err:
        print(err)

    ```

בואו נסביר את הקוד הזה:

- ראשית, אנו מייבאים את הספריות שאנחנו צריכים, כולל ספריית OpenAI, ספריית dotenv, המודול base64, וספריית Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- לאחר מכן, אנו יוצרים את הלקוח, שקורא את מפתח ה-API מהקובץ ``.env`` שלך.

    ```python
    # צור אובייקט OpenAI
    client = openai.OpenAI()
    ```

- בהמשך, אנו מייצרים את התמונה:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # הזן כאן את טקסט ההנחיה שלך
        size='1024x1024',
        n=1
    )
    ```

    דגמי `gpt-image` מחזירים את התמונה כמחרוזת **base64** ב- `data[0].b64_json`. אנו מפענחים אותה לבייטים וכותבים אותה לקובץ — אין URL להורדה.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- לבסוף, אנו פותחים את התמונה ומשתמשים בצופה התמונה הרגיל כדי להציג אותה:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### פרטים נוספים על יצירת התמונה

בואו נבחן את הפרמטרים של `images.generate`:

- **model**, הוא דגם התמונה לשימוש (לדוגמה `gpt-image-1`).
- **prompt**, הוא תיאור הטקסט המשמש ליצירת התמונה. כאן זה "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, הוא גודל התמונה שנוצרה (לדוגמה `1024x1024`, `1536x1024`, `1024x1536`, או `"auto"`).
- **n**, הוא מספר התמונות שנוצרו. כאן אנו מייצרים אחת.

> דגמי תמונה אינם מקבלים פרמטר `temperature` — זהו בקרה ליצירת טקסט. כדי לקבל גיוון, קרא ל-API שוב; כדי לצמצם את הגיוון, עשה את התיאור שלך מדויק יותר.

## יכולות נוספות ביצירת תמונה

ראית כיצד ליצור תמונה בכמה שורות פייתון. דגמי `gpt-image` יכולים גם **לערוך** תמונה קיימת. על ידי מתן תמונה, **מסכה** אופציונלית (המסמנת את האזור לשינוי), ותיאור, ניתן לשנות חלק מהתמונה — למשל, להוסיף כובע לארנב שלנו.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# העריכות מוחזרות גם כ-base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

התמונה הבסיסית מכילה רק את הארנב; התמונה הסופית כוללת את הכובע על הארנב.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
